In [1]:
import time
import math

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken
import itertools

torch.backends.cudnn.benchmark = True

# Softmax and angular attention

In [2]:
# A. Softmax MHA

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att  = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj= nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )


    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_query(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        K = self.W_key(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        V = self.W_value(x).view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T,:T], float('-inf'))
        W = torch.softmax(scores, dim=-1)
        W = self.dropout(W)

        out = (W @ V).transpose(1,2).reshape(B, T, -1)
        return self.out_proj(out)

# B. Angular Attention
class AngularBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att  = AngularAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

class AngularAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj= nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_query(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        K = self.W_key(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        V = self.W_value(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        Q = F.normalize(Q, dim=-1, eps=1e-6)
        K = F.normalize(K, dim=-1, eps=1e-6)

        cos_sim = (Q @ K.transpose(-2,-1)).clamp(-0.999,0.999)
        scores  = 1 - torch.acos(cos_sim)/torch.pi
        scores.masked_fill_(self.mask[:T,:T], 0.0)
        W = scores.clamp(min=1e-6).pow(18.0)
        W = W / (W.sum(-1,keepdim=True)+1e-6)
        W = self.dropout(W)

        out = (W @ V).transpose(1,2).reshape(B, T, -1)
        return self.out_proj(out)

# Non-differentiable RACEAttention with no causal masking: DO NOT RUN THIS!
* This is the older approach, much worse than vanilla and angular attention because of nonsmooth bucketing

In [ ]:
# C. RACE

#=====================================================================
# Fast, loop‑free ACE sketch works for S = H.N_M heads
#=====================================================================
class BatchedACE(nn.Module):
    """
    planes_flat : [S, L, K, d_k]   (float16)
    A_flat      : [S*L, R]         (int16 counters)
    B_flat      : [S*L, R, d_k]    (float16 value sums)

    In some part of the code, I called S as NH.

    This is the non-differentiable RACE
    """
    def __init__(self, d_k, K, L, N_M, H, device, alpha=0.25):
        super().__init__()
        self.K, self.L   = K, L
        self.N_M, self.H = N_M, H
        self.d_k, self.R = d_k, 1 << K
        self.alpha       = alpha

        NH  = H * N_M         # total (ensemble × head)
        HL  = NH * L          # flattened rows

        # 1. random hyperplanes  … store as float16 to cut BW
        planes = torch.randn(N_M, H, L, K, d_k, device=device).half()
        self.register_buffer("planes_flat", planes.reshape(NH, L, K, d_k))

        # 2. bit‑weights  [1,1,1,K]  for packing LSH bits → bucket id
        bw = (2 ** torch.arange(K, device=device)).view(1, 1, 1, K)
        self.register_buffer("bit_weights", bw)

        # 3. flattened counter / sum buffers
        self.register_buffer("A_flat",
                             torch.zeros(HL, self.R, dtype=torch.int16, device=device))
        self.register_buffer("B_flat",
                             torch.zeros(HL, self.R, d_k, dtype=torch.float16, device=device))

    # ----------------------------------------------------------------------
    @torch.no_grad()
    def clear(self):
        self.A_flat.zero_()
        self.B_flat.zero_()

    # ----------------------------------------------------------------------
    @torch.no_grad()
    def add_batch(self, Khf, Vhf):
        """
        Khf, Vhf : [S, N, d_k]   (S = H.N_M ,  N = B·T)
        """
        NH, N, _ = Khf.shape
        L, K, d_k = self.L, self.K, self.d_k
        HL = NH * L

        # 1. hash every token once  --> buckets [NH, N, L]
        proj    = torch.einsum('pnd,plkd->pnlk', Khf.half(), self.planes_flat)
        bits    = proj > 0
        buckets = (bits * self.bit_weights).sum(-1).long()          # [NH,N,L]

        # 2. stochastic token subsample (alpha fraction kept)
        keep = torch.rand(N, device=Khf.device) < self.alpha        # [N]
        buckets   = buckets[:, keep]                                # [NH,N_sub,L]
        Vhf_sub   = Vhf[:, keep].half()                             # [NH,N_sub,d_k]
        N_sub     = buckets.size(1)

        # 3. flatten (NH,L) --> HL and scatter once
        idx_flat = buckets.permute(0, 2, 1).reshape(HL, N_sub)      # [HL,N_sub]

        #   counters
        self.A_flat.scatter_add_(
            1, idx_flat,
            torch.ones_like(idx_flat, dtype=self.A_flat.dtype)
        )

        #   value sums
        Vflat = (Vhf_sub.unsqueeze(1)                               # [NH,1,N_sub,d_k]
                           .expand(NH, L, N_sub, d_k)
                           .reshape(HL, N_sub, d_k))                # [HL,N_sub,d_k]
        self.B_flat.scatter_add_(
            1,
            idx_flat.unsqueeze(-1).expand(-1, -1, d_k),             # [HL,N_sub,d_k]
            Vflat
        )

    # ----------------------------------------------------------------------
    def query(self, Qhf, row_choice: int = 0):
        """
        Qhf : [NH, N, d_k]    -->   returns [NH, N, d_k]
        Fast path: estimate with **one** sketch row (row_choice).
        """
        NH, N, _ = Qhf.shape
        row = max(0, min(row_choice, self.L - 1))                   # clamp

        # 1) hash queries once  → buckets_row [NH,N]
        proj = torch.einsum('pnd,plkd->pnlk', Qhf.half(), self.planes_flat)
        bits = proj > 0
        buckets_row = (bits * self.bit_weights).sum(-1)[:, :, row].long()

        # 2) gather counts & sums for that row
        #    self.A_flat row slice is [NH, R] thanks to flattening:
        row_offset = row + torch.arange(NH, device=Qhf.device) * self.L
        A_row = self.A_flat[row_offset]               # [NH, R]
        B_row = self.B_flat[row_offset]               # [NH, R, d_k]

        cnt = A_row.gather(1, buckets_row)            # [NH, N]
        val = B_row.gather(
                1,
                buckets_row.unsqueeze(-1).expand(-1, -1, self.d_k)
              )                                       # [NH, N, d_k]

        return val.float() / (cnt.unsqueeze(-1).float() + 1e-6)


#=====================================================================
# RACEAttention with the new BatchedACE
#=====================================================================
class RACEAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,
                 num_heads, qkv_bias=False, L=2, K=3, N_M=1,
                 alpha=0.25, device='cuda'):
        super().__init__()
        assert d_in % num_heads == 0
        self.H   = num_heads
        self.d_k = d_in // num_heads
        self.N_M = N_M

        # Linear projections
        self.q_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.k_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.v_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.out    = nn.Linear(d_in, d_out)
        self.dropout = nn.Dropout(dropout)

        # Fused ACE sketch
        self.ace = BatchedACE(
            d_k=self.d_k, K=K, L=L, N_M=N_M, H=num_heads,
            alpha=alpha, device=device
        )

    # ------------------------------------------------------------------
    def forward(self, x):
        B, T, _ = x.shape
        N = B * T

        # project & reshape → [B,T,H,d_k]
        Q = self.q_proj(x).view(B, T, self.H, self.d_k)
        K = self.k_proj(x).view(B, T, self.H, self.d_k)
        V = self.v_proj(x).view(B, T, self.H, self.d_k)

        # collapse heads --> [H,N,d_k]   then tile N_M ensembles
        def collapse(t):
            return t.permute(2, 0, 1, 3).reshape(self.H, N, self.d_k)
        Khf = collapse(K).repeat(self.N_M, 1, 1)
        Vhf = collapse(V).repeat(self.N_M, 1, 1)
        Qhf = collapse(Q).repeat(self.N_M, 1, 1)

        # build + query sketch
        with torch.no_grad():
            self.ace.clear()
            self.ace.add_batch(Khf, Vhf)
            
        out = self.ace.query(Qhf)                     # [NH,N,d_k]

        # reshape back → [B,T,H,d_k] then merge heads
        out = out.view(self.N_M, self.H, B, T, self.d_k).mean(0)  # [H,B,T,d_k]
        out = out.permute(1, 2, 0, 3).reshape(B, T, -1)           # [B,T,d_in]

        return self.dropout(self.out(out))



class RACEBlock(nn.Module):
    
    def __init__(self, cfg, device):
        super().__init__()
        self.att = RACEAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"], num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"], 
            L=2, K=3, N_M=2, alpha=0.1, device=device
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                        nn.GELU(),
                        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                     )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x); x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x); x = self.drop(x) + h
        return x

# Differentiable RACEAttention: DO NOT USE THIS!
* No half-precision for the time being
* Soft (differentiable) bucketization
* No sampling parameter $\alpha$ at this point

In [18]:
class BatchedACE(nn.Module):
    def __init__(self, d_k, K, L, N_M, H, device='cuda'):
        super().__init__()
        self.K, self.L, self.d_k = K, L, d_k
        self.R = 1 << K

        # full-precision planes
        planes = torch.randn(L, K, d_k, device=device)
        self.register_buffer('planes_flat', planes)

        # bucket prototypes
        corners = torch.tensor(
            list(itertools.product([-1.0, +1.0], repeat=K)),
            device=device
        )
        self.register_buffer('bucket_protos', corners)

        # # learnable temperature
        # self.logit_temp = nn.Parameter(torch.log(torch.tensor(1.0)))

    def forward(self, Khf, Vhf, Qhf):
        # Khf, Vhf, Qhf: [M, B, T, H, d_k]
        # we assume they are already shaped as [M,B,T,H,d_k]
        M, B, T, H, d_k = Khf.shape
        L, K, _ = self.planes_flat.shape

        # temp is basically the scaling factor in attention. One can make it learnable
        # to further improve the performance
        temp = math.sqrt(d_k)
        # temp = self.logit_temp.exp().clamp(1e-2, 10.0) # uncomment when you make temp learnable
        

        # 1) soft-hash keys --> [M,B,T,H,L,R]
        proj_K   = torch.einsum('mbthd,lkd->mbthlk', Khf, self.planes_flat)
        softbits = torch.tanh(proj_K.clamp(-10,10)) / temp
        logits_K = torch.einsum('mbthlk,rk->mbthlr', softbits, self.bucket_protos)
        probs_K  = torch.softmax(logits_K, dim=-1)

        # 2) causal counts & sums
        V_ext  = Vhf.unsqueeze(-2).unsqueeze(-2)  # [M,B,T,H,1,1,d_k]
        A_pref = probs_K.cumsum(dim=2)            # prefix over T
        B_pref = (probs_K.unsqueeze(-1) * V_ext).cumsum(dim=2)
        E_pref = B_pref / (A_pref.unsqueeze(-1) + 1e-6)

        # 3) soft-hash queries --> [M,B,T,H,L,R]
        proj_Q   = torch.einsum('mbthd,lkd->mbthlk', Qhf, self.planes_flat)
        bits_Q   = torch.tanh(proj_Q.clamp(-10,10)) / temp
        logits_Q = torch.einsum('mbthlk,rk->mbthlr', bits_Q, self.bucket_protos)
        probs_Q  = torch.softmax(logits_Q, dim=-1)

        # 4) expected‐value lookup --> [M,B,T,H,d_k]
        mul      = probs_Q.unsqueeze(-1) * E_pref    # [M,B,T,H,L,R,d_k]
        sum_L    = mul.sum(dim=4)                    # sum over L --> [M,B,T,H,R,d_k]
        out_pref = sum_L.sum(dim=4)                  # sum over R --> [M,B,T,H,d_k]

        return out_pref  # [M,B,T,H,d_k]

class RACEAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout,
                 num_heads, qkv_bias=False, L=2, K=3, N_M=1, device='cuda'):
        super().__init__()
        assert d_in % num_heads == 0
        self.H   = num_heads
        self.d_k = d_in // num_heads
        self.M   = N_M

        self.q_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.k_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.v_proj = nn.Linear(d_in, d_in, bias=qkv_bias)
        self.out    = nn.Linear(d_in, d_out)
        self.drop   = nn.Dropout(dropout)

        self.ace = BatchedACE(self.d_k, K, L, N_M, num_heads, device=device)

    def forward(self, x):
        B, T, _ = x.shape
        H, d_k, M = self.H, self.d_k, self.M

        # 1) project & reshape for ACE
        Q = self.q_proj(x).view(B, T, H, d_k)
        K = self.k_proj(x).view(B, T, H, d_k)
        V = self.v_proj(x).view(B, T, H, d_k)

        # shape --> [M, B, T, H, d_k] by explicit unsqueeze
        def pack(Z):
            Zm = Z.unsqueeze(0).expand(M, -1, -1, -1, -1)
            return Zm

        Khf = pack(K)
        Vhf = pack(V)
        Qhf = pack(Q)

        # 2) run ACE
        out_hm = self.ace(Khf, Vhf, Qhf)  # [M,B,T,H,d_k]

        # 3) average ensembles & merge heads
        out = out_hm.mean(dim=0)          # [B,T,H,d_k]
        out = out.permute(0,2,1,3).reshape(B, T, H * d_k)

        # 4) final proj + dropout
        return self.drop(self.out(out))


class RACEBlock(nn.Module):
    def __init__(self, cfg, device='cuda'):
        super().__init__()
        self.att   = RACEAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"], qkv_bias=cfg["qkv_bias"],
            L=2, K=3, N_M=2, device=device
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop(x) + h

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x) + h

        return x

# Faster, Differentiable RACEAttention: USE THIS!
* Two big GEMMs
* Best so far
* Shared hyperplanes can be used
* Causal masking can be further optimized (??)

In [3]:
class BatchedACE(nn.Module):
    """
    Causal (LM) BatchedACE that optionally uses a single shared sketch
    or independent sketches per ensemble.
    Inputs:
      Khf, Vhf, Qhf : [M, B, T, H, d_k]
    """
    def __init__(self, d_k, K, L, M, device='cuda', share_planes: bool = False):
        super().__init__()
        self.d_k, self.K, self.L, self.M = d_k, K, L, M
        self.R = 1 << K
        self.share_planes = share_planes

        if share_planes:
            # Shared planes [L, K, d_k] --> [d_k, (L*K)]
            planes = torch.randn(L, K, d_k, device=device)
            self.register_buffer(
              'planes_T',
              planes.view(L*K, d_k).T
            )  # [d_k, L*K]
        else:
            # Independent planes [M, L, K, d_k] --> [M, d_k, (L*K)]
            planes = torch.randn(M, L, K, d_k, device=device)
            planes = planes.view(M, L*K, d_k).transpose(1,2)
            self.register_buffer('planes_T', planes)
            # planes_T: [M, d_k, L*K]

        # flatten protos [R, K] → [K, R]
        corners = torch.tensor(
            list(itertools.product([-1., +1.], repeat=K)),
            device=device
        )
        self.register_buffer('protos_T', corners.T)  # [K, R]

    def forward(self, Khf, Vhf, Qhf):
        # [M, B, T, H, d_k]
        M, B, T, H, dk = Khf.shape
        assert M == self.M and dk == self.d_k
        scale = math.sqrt(dk)

        # collapse dims → [?, T, d_k]
        if self.share_planes:
            # full collapse over M·B·H
            N = M * B * H
            Kh2 = Khf.permute(0,1,3,2,4).reshape(N, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).reshape(N, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).reshape(N, T, dk)

            # single GEMM per projection
            projK = Kh2 @ self.planes_T                # [N, T, L*K]
            projQ = Qh2 @ self.planes_T                # [N, T, L*K]
        else:
            # collapse only batch+head per ensemble: [M, BH, T, d_k]
            BH = B * H
            Kh2 = Khf.permute(0,1,3,2,4).reshape(M, BH, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).reshape(M, BH, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).reshape(M, BH, T, dk)

            # one batched GEMM across ensembles
            projK = torch.einsum('mbtd, mds -> mbts', Kh2, self.planes_T)
            projQ = torch.einsum('mbtd, mds -> mbts', Qh2, self.planes_T)
            # merge M,BH → N
            projK = projK.reshape(M*BH, T, self.L*self.K)
            projQ = projQ.reshape(M*BH, T, self.L*self.K)
            V2    = V2.reshape(M*BH, T, dk)
            N     = M * BH

        # reshape --> [N, T, L, K]
        projK = projK.view(N, T, self.L, self.K)
        projQ = projQ.view(N, T, self.L, self.K)

        # soft‑hash & bucket‑protos
        logitsK = (projK.tanh().div(scale) @ self.protos_T)  # [N, T, L, R]
        probsK  = F.softmax(logitsK, dim=-1)
        logitsQ = (projQ.tanh().div(scale) @ self.protos_T)
        probsQ  = F.softmax(logitsQ, dim=-1)

        # 2) causal prefix sums
        A_pref = probsK.cumsum(dim=1)                                    # [N, T, L, R]
        B_pref = (probsK.unsqueeze(-1) * V2.unsqueeze(2).unsqueeze(3))\
                   .cumsum(dim=1)                                       # [N, T, L, R, d_k]
        E_pref = B_pref.div(A_pref.unsqueeze(-1).add(1e-6))              # [N, T, L, R, d_k]

        # 3) lookup via one batched bmm
        S      = self.L * self.R
        pQ_flat= probsQ.view(N, T, S)                                   # [N, T, S]
        E_flat = E_pref.view(N, T, S, dk)[:, -1]                        # [N, S, d_k]
        out2   = torch.bmm(pQ_flat, E_flat)                             # [N, T, d_k]

        # 4) reshape back → [M, B, T, H, d_k]
        out = out2.view(M, B, H, T, dk).permute(0,1,3,2,4)
        return out


class RACEAttention(nn.Module):
    """Multi‑head wrapper around BatchedACE."""
    def __init__(self, d, h, K=3, L=2, M=2, drop=0.1,
                 qkv_bias=False, device='cuda'):
        super().__init__()
        assert d % h == 0
        self.h, self.d_k, self.M = h, d//h, M
        self.q = nn.Linear(d, d, bias=qkv_bias)
        self.k = nn.Linear(d, d, bias=qkv_bias)
        self.v = nn.Linear(d, d, bias=qkv_bias)
        self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        self.ace = BatchedACE(
                       self.d_k, K, L, M, device=device
                   )

    def forward(self, x):
        B, T, _ = x.shape
        # split heads
        Q = self.q(x).view(B, T, self.h, self.d_k)
        K = self.k(x).view(B, T, self.h, self.d_k)
        V = self.v(x).view(B, T, self.h, self.d_k)
        # pack M ensembles
        pack = lambda z: z.unsqueeze(0).expand(self.M, -1, -1, -1, -1)
        Khf, Vhf, Qhf = pack(K), pack(V), pack(Q)
        # single-shot causal ACE
        out_h = self.ace(Khf, Vhf, Qhf)       # [M, B, T, H, d_k]
        # merge ensembles & heads
        out   = out_h.mean(0).permute(0,2,1,3).reshape(B, T, -1)
        return self.drop(self.o(out))


class RACEBlock(nn.Module):
    def __init__(self, cfg, device='cuda'):
        super().__init__()
        self.att = RACEAttention(
                       d     = cfg["emb_dim"],
                       h     = cfg["n_heads"],
                       K     = cfg.get("K", 1),
                       L     = cfg.get("L", 1),
                       M     = cfg.get("M", 1),
                       drop  = cfg["drop_rate"],
                       qkv_bias    = cfg.get("qkv_bias", False),
                       device      = device
                   )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                       nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                       nn.GELU(),
                       nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                   )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x)
        return self.drop(x) + h

# LM

In [4]:
class LMModel(nn.Module):
    def __init__(self, cfg, attn_type="race", device="cuda"):
        super().__init__()
        self.tok_emb    = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb    = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb   = nn.Dropout(cfg["drop_rate"])
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

        if attn_type == "race":
            AttnBlock = lambda c: RACEBlock(c, device)
        elif attn_type == "gpt":
            AttnBlock = TransformerBlock
        elif attn_type == "angular":
            AttnBlock = AngularBlock
        else:
            raise ValueError(f"Unknown attn_type={attn_type}")

        self.blocks = nn.Sequential(
            *[AttnBlock(cfg) for _ in range(cfg["n_layers"])]
        )

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)[None]
        x = self.tok_emb(x) + self.pos_emb(pos)
        x = self.drop_emb(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# Dataset, training and validation loops

In [5]:
#=====================================================================
# PTB dataset & loader
#=====================================================================
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, L, S):
        self.inputs, self.targets = [], []
        ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(ids)-L, S):
            self.inputs .append(torch.tensor(ids[i:i+L]))
            self.targets.append(torch.tensor(ids[i+1:i+L+1]))
    def __len__(self): return len(self.inputs)
    def __getitem__(self,i): return self.inputs[i], self.targets[i]

def create_dataloader(txt, bs, L, S):
    tok = tiktoken.get_encoding("gpt2")
    ds  = GPTDatasetV1(txt, tok, L, S)
    return DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True)

#=====================================================================
# run_experiment
#=====================================================================

def run_experiment(attn_types, epochs=5, lr=1e-5, wd=0.01):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = {
      "vocab_size":50257, "context_length":64,
      "emb_dim":512, "n_heads":8, "n_layers":4,
      "drop_rate":0.1, "qkv_bias":False
    }


    for kind in attn_types:
        model = LMModel(cfg, attn_type=kind, device=device).to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)


        print(f"\n=== Training {kind.upper()} for {epochs} epochs ===")
        for ep in range(1, epochs+1):
            # --- TR ---
            t0 = time.time()
            model.train()
            tl = ta = 0.0
            for x,y in train_loader:
                opt.zero_grad()
                logits = model(x.to(device))
                loss   = F.cross_entropy(logits.flatten(0,1), y.to(device).flatten())
                acc    = (logits.argmax(-1)==y.to(device)).float().mean().item()
                loss.backward(); 
                opt.step()
                tl += loss.item(); ta += acc
            train_time = time.time() - t0

            tr_l = tl/len(train_loader)
            tr_a = ta/len(train_loader)
            tr_p = math.exp(tr_l)

            # --- VAL---
            t1 = time.time()
            model.eval()
            vl = va = 0.0
            with torch.no_grad():
                for x,y in val_loader:
                    logits = model(x.to(device))
                    loss   = F.cross_entropy(logits.flatten(0,1), y.to(device).flatten())
                    acc    = (logits.argmax(-1)==y.to(device)).float().mean().item()
                    vl += loss.item(); va += acc
            val_time = time.time() - t1

            va_l = vl/len(val_loader)
            va_a = va/len(val_loader)
            va_p = math.exp(va_l)

            print(
                f"Ep{ep:2d} | "
                f"train loss {tr_l:.3f}, acc {tr_a:.3f}, ppl {tr_p:.2f} "
                f"(train_time {train_time:.1f}s) | "
                f"val loss {va_l:.3f}, acc {va_a:.3f}, ppl {va_p:.2f} "
                f"(val_time {val_time:.1f}s)"
            )


# PTB comes with train/validation/test splits, and each row is a sentence.
ptb = load_dataset("ptb_text_only")

# Join all sentences into one long string per split:
tr_txt = " ".join(ptb["train"]["sentence"])
va_txt = " ".join(ptb["validation"]["sentence"])
# use ptb["test"] if you want a held-out test set

# Now create your dataloaders just as before:
train_loader = create_dataloader(tr_txt, bs=16, L=64, S=32)
val_loader   = create_dataloader(va_txt, bs=16, L=64, S=32)

README.md: 0.00B [00:00, ?B/s]

ptb_text_only.py: 0.00B [00:00, ?B/s]

The repository for ptb_text_only contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ptb_text_only.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split:   0%|          | 0/42068 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3761 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3370 [00:00<?, ? examples/s]

In [21]:
# Old differentiable BatchedACE with einsums
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.591, acc 0.169, ppl 728.29 (train_time 221.3s) | val loss 5.791, acc 0.199, ppl 327.32 (val_time 4.5s)
Ep 2 | train loss 5.618, acc 0.217, ppl 275.43 (train_time 221.7s) | val loss 5.450, acc 0.231, ppl 232.70 (val_time 4.5s)
Ep 3 | train loss 5.351, acc 0.238, ppl 210.74 (train_time 221.7s) | val loss 5.248, acc 0.246, ppl 190.24 (val_time 4.5s)
Ep 4 | train loss 5.167, acc 0.250, ppl 175.45 (train_time 222.3s) | val loss 5.107, acc 0.256, ppl 165.11 (val_time 4.5s)
Ep 5 | train loss 5.026, acc 0.259, ppl 152.26 (train_time 221.7s) | val loss 5.001, acc 0.265, ppl 148.60 (val_time 4.5s)
Ep 6 | train loss 4.911, acc 0.267, ppl 135.83 (train_time 222.0s) | val loss 4.914, acc 0.271, ppl 136.21 (val_time 4.5s)
Ep 7 | train loss 4.817, acc 0.274, ppl 123.57 (train_time 221.7s) | val loss 4.848, acc 0.277, ppl 127.52 (val_time 4.5s)
Ep 8 | train loss 4.736, acc 0.280, ppl 113.97 (train_time 222.0s) | val loss 4.789, acc 0.281, ppl 12

In [6]:
# Faster (GEMMs), differentiable BatchedACE with share_planes=True
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.581, acc 0.171, ppl 721.39 (train_time 197.6s) | val loss 5.775, acc 0.203, ppl 322.25 (val_time 4.0s)
Ep 2 | train loss 5.606, acc 0.220, ppl 271.97 (train_time 197.1s) | val loss 5.431, acc 0.234, ppl 228.36 (val_time 4.0s)
Ep 3 | train loss 5.331, acc 0.242, ppl 206.57 (train_time 196.5s) | val loss 5.213, acc 0.253, ppl 183.71 (val_time 4.0s)
Ep 4 | train loss 5.137, acc 0.255, ppl 170.26 (train_time 197.0s) | val loss 5.066, acc 0.264, ppl 158.49 (val_time 4.0s)
Ep 5 | train loss 4.990, acc 0.266, ppl 146.92 (train_time 197.0s) | val loss 4.957, acc 0.272, ppl 142.16 (val_time 4.0s)
Ep 6 | train loss 4.870, acc 0.274, ppl 130.38 (train_time 196.6s) | val loss 4.868, acc 0.279, ppl 130.02 (val_time 4.0s)
Ep 7 | train loss 4.772, acc 0.281, ppl 118.12 (train_time 196.4s) | val loss 4.796, acc 0.284, ppl 120.98 (val_time 4.0s)
Ep 8 | train loss 4.686, acc 0.286, ppl 108.46 (train_time 196.9s) | val loss 4.733, acc 0.289, ppl 11

In [17]:
# Faster (GEMMs), differentiable BatchedACE with share_planes=False
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.583, acc 0.170, ppl 722.63 (train_time 200.4s) | val loss 5.774, acc 0.203, ppl 321.85 (val_time 4.0s)
Ep 2 | train loss 5.605, acc 0.220, ppl 271.77 (train_time 198.9s) | val loss 5.426, acc 0.234, ppl 227.32 (val_time 4.0s)
Ep 3 | train loss 5.327, acc 0.242, ppl 205.74 (train_time 199.0s) | val loss 5.211, acc 0.252, ppl 183.35 (val_time 4.0s)
Ep 4 | train loss 5.134, acc 0.255, ppl 169.65 (train_time 198.7s) | val loss 5.065, acc 0.263, ppl 158.42 (val_time 4.0s)
Ep 5 | train loss 4.986, acc 0.266, ppl 146.41 (train_time 198.9s) | val loss 4.956, acc 0.272, ppl 141.99 (val_time 4.0s)
Ep 6 | train loss 4.867, acc 0.274, ppl 129.91 (train_time 199.0s) | val loss 4.865, acc 0.279, ppl 129.66 (val_time 4.0s)
Ep 7 | train loss 4.768, acc 0.281, ppl 117.64 (train_time 198.5s) | val loss 4.793, acc 0.284, ppl 120.66 (val_time 4.0s)
Ep 8 | train loss 4.682, acc 0.287, ppl 107.98 (train_time 198.7s) | val loss 4.729, acc 0.288, ppl 11

In [8]:
run_experiment(["gpt"],epochs=10)


=== Training GPT for 10 epochs ===
Ep 1 | train loss 6.590, acc 0.166, ppl 728.01 (train_time 137.8s) | val loss 5.807, acc 0.198, ppl 332.52 (val_time 3.1s)
Ep 2 | train loss 5.631, acc 0.216, ppl 278.97 (train_time 137.6s) | val loss 5.466, acc 0.227, ppl 236.44 (val_time 3.1s)
Ep 3 | train loss 5.365, acc 0.234, ppl 213.74 (train_time 137.5s) | val loss 5.265, acc 0.241, ppl 193.47 (val_time 3.1s)
Ep 4 | train loss 5.184, acc 0.246, ppl 178.36 (train_time 137.5s) | val loss 5.129, acc 0.251, ppl 168.79 (val_time 3.1s)
Ep 5 | train loss 5.045, acc 0.255, ppl 155.29 (train_time 137.5s) | val loss 5.022, acc 0.259, ppl 151.73 (val_time 3.1s)
Ep 6 | train loss 4.932, acc 0.262, ppl 138.61 (train_time 137.6s) | val loss 4.934, acc 0.266, ppl 139.00 (val_time 3.1s)
Ep 7 | train loss 4.835, acc 0.269, ppl 125.88 (train_time 137.5s) | val loss 4.866, acc 0.271, ppl 129.84 (val_time 3.1s)
Ep 8 | train loss 4.751, acc 0.275, ppl 115.74 (train_time 137.3s) | val loss 4.798, acc 0.277, ppl 121

In [9]:
run_experiment(["angular"],epochs=10)


=== Training ANGULAR for 10 epochs ===
Ep 1 | train loss 6.577, acc 0.167, ppl 718.44 (train_time 155.5s) | val loss 5.792, acc 0.200, ppl 327.80 (val_time 3.4s)
Ep 2 | train loss 5.629, acc 0.216, ppl 278.28 (train_time 155.4s) | val loss 5.460, acc 0.227, ppl 235.19 (val_time 3.4s)
Ep 3 | train loss 5.360, acc 0.235, ppl 212.63 (train_time 155.4s) | val loss 5.251, acc 0.243, ppl 190.74 (val_time 3.4s)
Ep 4 | train loss 5.172, acc 0.248, ppl 176.28 (train_time 155.5s) | val loss 5.105, acc 0.255, ppl 164.83 (val_time 3.4s)
Ep 5 | train loss 5.026, acc 0.258, ppl 152.34 (train_time 155.5s) | val loss 4.994, acc 0.263, ppl 147.46 (val_time 3.4s)
Ep 6 | train loss 4.908, acc 0.266, ppl 135.36 (train_time 155.3s) | val loss 4.906, acc 0.269, ppl 135.15 (val_time 3.4s)
Ep 7 | train loss 4.811, acc 0.273, ppl 122.82 (train_time 155.5s) | val loss 4.834, acc 0.275, ppl 125.74 (val_time 3.4s)
Ep 8 | train loss 4.727, acc 0.279, ppl 113.01 (train_time 155.3s) | val loss 4.775, acc 0.280, ppl

In [6]:
# Faster (GEMMs), differentiable BatchedACE with share_planes=False, K =1, L = 1, M = 1
run_experiment(["race"],epochs=10)


=== Training RACE for 10 epochs ===
Ep 1 | train loss 6.589, acc 0.168, ppl 726.86 (train_time 148.3s) | val loss 5.791, acc 0.199, ppl 327.46 (val_time 3.3s)
Ep 2 | train loss 5.616, acc 0.218, ppl 274.69 (train_time 146.8s) | val loss 5.445, acc 0.230, ppl 231.60 (val_time 3.3s)


KeyboardInterrupt: 